In [ ]:
import numpy as np
import scipy
from tueplots import bundles
import matplotlib.pyplot as plt
## setting matplotlib context
from cycler import cycler
from matplotlib.cm import get_cmap
cmap = get_cmap("tab10",8)
palette = [cmap(i) for i in range(8)]
rc = bundles.neurips2024(usetex=False)
rc.update({
    # Set the line/bar color cycle (this is what affects ax.plot)
    "axes.prop_cycle": cycler(color=palette),
    # Optional readability tweaks
    "legend.frameon": False,
    "axes.grid": False,
})


### Objective 1:  Analytical vs. Empirical Standard Deviations

In [ ]:
np.random.seed(0)

N = 1000
d = 500
N_train = int(N//2)
sigma = 1.0
eps = 1e-12
M = 2048

# Target Superset #
X = scipy.stats.norm.rvs(loc=0.0, scale=sigma, size=(N,d))

# Membership Matrix #
memberships = np.zeros((M, N), dtype=bool)
for r in range(M):
    memberships[r] = np.random.binomial(n=1,p=0.5,size=N).astype(bool)
    # idx = np.random.choice(N, size=N_train, replace=False)
    # memberships[r, idx] = True

# Computing Model Statistics #
model_means = (memberships @ X) / N_train  
stats      = model_means @ X.T            

mu_ins, mu_outs ,sigma_ins, sigma_outs = np.zeros(N), np.zeros(N), np.zeros(N), np.zeros(N)
for n in range(N):
    curr_stats, curr_indices =  stats[:,n], memberships[:,n]
    in_stats, out_stats = curr_stats[curr_indices], curr_stats[~curr_indices]
    mu_ins[n], mu_outs[n], sigma_ins[n], sigma_outs[n] = np.mean(in_stats), np.mean(out_stats), np.std(in_stats), np.std(out_stats)

In [ ]:
X_norm  = np.linalg.norm(X, axis=1)     
analytical_sigma_out = sigma * X_norm / np.sqrt(N_train)
analytical_sigma_in  = sigma * X_norm * np.sqrt(N_train - 1) / N_train

# fpc_in  = 1.0 - (N_train - 1) / (N - 1)
# fpc_out = 1.0 - N_train       / (N - 1)

fpc_in = fpc_out = 1.0 - N_train       / (N - 1)

sigma_ins_corr  = sigma_ins  / np.sqrt(fpc_in)
sigma_outs_corr = sigma_outs / np.sqrt(fpc_out)

print("── IN std comparison (mean over samples) ──")
print(f"  analytical:          {analytical_sigma_in.mean():.6f}")
print(f"  empirical (raw):     {sigma_ins.mean():.6f}")
print(f"  empirical / √fpc_in: {sigma_ins_corr.mean():.6f}")
print()
print("── OUT std comparison (mean over samples) ──")
print(f"  analytical:           {analytical_sigma_out.mean():.6f}")
print(f"  empirical (raw):      {sigma_outs.mean():.6f}")
print(f"  empirical / √fpc_out: {sigma_outs_corr.mean():.6f}")

In [ ]:
rc = bundles.neurips2024(usetex=False,rel_width=0.5)
rc.update({
    # Set the line/bar color cycle (this is what affects ax.plot)
    "axes.prop_cycle": cycler(color=palette),
    # Optional readability tweaks
    "legend.frameon": False,
    "axes.grid": False,
})

with plt.rc_context(rc):
    fig, ax = plt.subplots(1, 2, sharex="row", sharey="row")

    plots = [
        ("OUT", analytical_sigma_out, sigma_outs, sigma_outs_corr),
        ("IN",  analytical_sigma_in,  sigma_ins,  sigma_ins_corr),
    ]

    for i, (label, analytical, empirical, corrected) in enumerate(plots):
        # sort by analytical std for a clean line plot
        order = np.argsort(analytical)

        ax[i].plot(
            analytical[order],
            color="r",
            label="Analytical",
            lw=1.5,
            
        )
        ax[i].plot(
            empirical[order],
            label="W/ LiRA",
            lw=1.5,
            ls="--",
            alpha=0.65
        )
        ax[i].plot(
            corrected[order],
            label="post-FPC",
            lw=1.5,
            ls=":",
            alpha=0.65
        )

        ax[i].set_title(rf"{label}")
        ax[i].set_xlabel(r"$X$s")
        ax[i].set_box_aspect(1)

    ax[0].set_ylabel(r"$\sigma$")

    h, l = ax[0].get_legend_handles_labels()
    fig.legend(
        h, l,
        loc="lower center",
        bbox_to_anchor=(0.55, -0.15),
        ncol=3,
        frameon=False,
    )
    plt.savefig("stdev_pre_post_fpc_lira.pdf", bbox_inches="tight")
    plt.show()

In [ ]:
# Ratio: corrected_empirical / analytical  (should be ≈ 1.0 if FPC rule is accurate)
ratio_in  = sigma_ins_corr  / analytical_sigma_in
ratio_out = sigma_outs_corr / analytical_sigma_out

# Also check raw compression ratio vs predicted compression from FPC
raw_ratio_in  = sigma_ins  / analytical_sigma_in   # should be ≈ √fpc_in
raw_ratio_out = sigma_outs / analytical_sigma_out  # should be ≈ √fpc_out

print("── Does empirical compression match FPC prediction? ──")
print(f" (IN) Raw Ratio:  {raw_ratio_in.mean():.6f}  (expected √fpc_in  = {np.sqrt(fpc_in):.6f})")
print(f" (OUT) Raw Ratio: {raw_ratio_out.mean():.6f}  (expected √fpc_out = {np.sqrt(fpc_out):.6f})")
print()
print("── After FPC correction (corrected/analytical, should be ≈ 1.0) ──")
print(f"  IN  — mean: {ratio_in.mean():.6f},  std: {ratio_in.std():.6f},  max|err|: {np.abs(ratio_in - 1).max():.6f}")
print(f"  OUT — mean: {ratio_out.mean():.6f},  std: {ratio_out.std():.6f},  max|err|: {np.abs(ratio_out - 1).max():.6f}")

In [ ]:
rc = bundles.neurips2024(usetex=False, rel_width=0.5)
rc.update({
    # Set the line/bar color cycle (this is what affects ax.plot)
    "axes.prop_cycle": cycler(color=palette),
    # Optional readability tweaks
    "legend.frameon": False,
    "axes.grid": False,
})

with plt.rc_context(rc):
    fig, ax = plt.subplots(1,2, sharey="row")

    ax[0].hist(raw_ratio_in, bins=25, density=True, alpha=0.65)
    ax[0].axvline(np.sqrt(fpc_in), color="red",  lw=1.5, ls="--", label=rf"$\sqrt{{\mathrm{{FPC}}}}$ = {np.sqrt(fpc_in):.4f}")
    ax[0].set_title("IN")
    ax[0].set_xlabel(r"$\dfrac{{\sigma_{emp}}}{{\sigma_{ana}}}$")
    ax[0].set_box_aspect(1)
    # ax[0].legend()

    ax[1].hist(raw_ratio_out, bins=25, density=True, alpha=0.65)
    ax[1].axvline(np.sqrt(fpc_out), color="red",  lw=1.5, ls="--", label=rf"$\sqrt{{\mathrm{{FPC}}}}$ = {np.sqrt(fpc_out):.4f}")
    ax[1].set_title("OUT")
    ax[1].set_xlabel(r"$\dfrac{{\sigma_{emp}}}{{\sigma_{ana}}}$")
    ax[1].set_box_aspect(1)
    # ax[1].legend()

    h, l = ax[0].get_legend_handles_labels()
    fig.legend(
        h, l,
        loc="lower center",
        bbox_to_anchor=(0.55, -0.15),
        ncol=3,
        frameon=False,
    )

    plt.savefig("fpc_compression_ratio.pdf", bbox_inches="tight")
    plt.show()
